In [1]:
import geopandas as gpd

In [2]:
print("Loading boundaries...")
boundaries_gdf = gpd.read_file("geoBoundaries-IND-ADM2_simplified.geojson")
print(f"✅ Boundaries loaded: {len(boundaries_gdf)} administrative zones found.")

print("\nLoading health facilities (103.2 MB file, please wait a moment)...")
hospitals_gdf = gpd.read_file("health_facilities.geojson")
print(f"✅ Health facilities loaded: {len(hospitals_gdf)} facilities found.")

print("\nLoading cyclone tracks (263.7 MB file, this will take ~30-60 seconds)...")
tracks_gdf = gpd.read_file("ibtracs_ALL_list_v04r01_lines.geojson")
print(f"✅ Cyclone tracks loaded: {len(tracks_gdf)} storm tracks found.")

# Verify Coordinate Reference Systems (CRS)
print("\n--- CRS Verification ---")
print("Boundaries CRS:", boundaries_gdf.crs)
print("Hospitals CRS :", hospitals_gdf.crs)
print("Tracks CRS    :", tracks_gdf.crs)

Loading boundaries...
✅ Boundaries loaded: 735 administrative zones found.

Loading health facilities (103.2 MB file, please wait a moment)...
✅ Health facilities loaded: 142629 facilities found.

Loading cyclone tracks (263.7 MB file, this will take ~30-60 seconds)...
✅ Cyclone tracks loaded: 713155 storm tracks found.

--- CRS Verification ---
Boundaries CRS: EPSG:4326
Hospitals CRS : EPSG:4326
Tracks CRS    : EPSG:4326


In [3]:
boundaries_gdf.head()

,shapeName,shapeISO,shapeID,shapeGroup,shapeType,geometry
0,Ashoknagar,,76128533B75548370501185,IND,ADM2,"POLYGON ((78.17491 24.84254, 78.16886 24.83734..."
1,Raisen,,76128533B57893545331548,IND,ADM2,"POLYGON ((77.38167 23.07004, 77.37253 23.06423..."
2,Chhindwara,,76128533B70646408240587,IND,ADM2,"POLYGON ((79.23988 22.79135, 79.24257 22.80093..."
3,Betul,,76128533B82559220423608,IND,ADM2,"POLYGON ((78.27229 22.39973, 78.26843 22.39477..."
4,Hoshangabad,,76128533B45314020251888,IND,ADM2,"POLYGON ((78.03027 22.79953, 78.02645 22.80026..."


In [4]:
hospitals_gdf.head()

,id,name,name_en,name_hi,amenity,building,healthcare,healthcare_speciality,operator_type,capacity_persons,...,adm1_pcode,adm1_name,adm2_pcode,adm2_name,adm3_pcode,adm3_name,adm4_pcode,adm4_name,name_latin,geometry
0,node/11384942485,YSR Public Hospital,NaN,NaN,hospital,NaN,hospital,NaN,NaN,None,...,1811400B84418948762846,Andhra Pradesh,76128533B59405966069108,Srikakulam,7132399B23766291644212,Srikakulam,None,None,YSR Public Hospital,POINT (83.90254 18.30554)
1,node/9903926780,NaN,NaN,NaN,hospital,NaN,hospital,NaN,NaN,None,...,1811400B84418948762846,Andhra Pradesh,76128533B59405966069108,Srikakulam,7132399B23766291644212,Srikakulam,None,None,NaN,POINT (83.90175 18.3051)
2,node/7786364882,Tirumala Maternity & Surgical Hospital,NaN,NaN,hospital,NaN,NaN,NaN,NaN,None,...,1811400B84418948762846,Andhra Pradesh,76128533B59405966069108,Srikakulam,7132399B23766291644212,Srikakulam,None,None,Tirumala Maternity & Surgical Hospital,POINT (83.90314 18.30162)
3,node/6798036388,Vijaya Harsha Thalli Pillala Hospital,NaN,NaN,hospital,NaN,NaN,NaN,NaN,None,...,1811400B84418948762846,Andhra Pradesh,76128533B59405966069108,Srikakulam,7132399B23766291644212,Srikakulam,None,None,Vijaya Harsha Thalli Pillala Hospital,POINT (83.90327 18.30147)
4,node/7786364852,Dr. S.Jyothi Prakash Raju,NaN,NaN,doctors,NaN,NaN,NaN,NaN,None,...,1811400B84418948762846,Andhra Pradesh,76128533B59405966069108,Srikakulam,7132399B23766291644212,Srikakulam,None,None,Dr. S.Jyothi Prakash Raju,POINT (83.89733 18.30855)


In [5]:
tracks_gdf.head()

,SID,NUMBER,BASIN,SUBBASIN,ISO_TIME,NATURE,LAT,LON,WMO_WIND,WMO_PRES,geometry
0,1842298N11080,1,North India,Bay of Bengal,1842-10-25 03:00:00,Not reported,10.9,80.3,NaN,NaN,"LINESTRING (80.30005 10.9, 79.80005 10.9)"
1,1842298N11080,1,North India,Bay of Bengal,1842-10-25 06:00:00,Not reported,10.9,79.8,NaN,NaN,"LINESTRING (79.80005 10.9, 79.40002 10.8)"
2,1842298N11080,1,North India,Bay of Bengal,1842-10-25 09:00:00,Not reported,10.8,79.4,NaN,NaN,"LINESTRING (79.40002 10.8, 78.90002 10.8)"
3,1842298N11080,1,North India,Bay of Bengal,1842-10-25 12:00:00,Not reported,10.8,78.9,NaN,NaN,"LINESTRING (78.90002 10.8, 78.40002 10.8)"
4,1842298N11080,1,North India,Bay of Bengal,1842-10-25 15:00:00,Not reported,10.8,78.4,NaN,NaN,"LINESTRING (78.40002 10.8, 77.90002 10.8)"


In [6]:
print("Starting Phase 2: Spatial Engine Setup...\n")

target_states = ['West Bengal', 'Odisha']
wb_odisha_hospitals = hospitals_gdf[hospitals_gdf['adm1_name'].isin(target_states)].copy()
print(f"✅ Filtered Hospitals: {len(wb_odisha_hospitals)} facilities isolated in WB and Odisha.")

bob_tracks = tracks_gdf[tracks_gdf['SUBBASIN'] == 'Bay of Bengal'].copy()
print(f"✅ Filtered Tracks: {len(bob_tracks)} historical storm segments isolated")

print("\nReprojecting coordinates to metric... (this takes a few seconds)")
hospitals_metric = wb_odisha_hospitals.to_crs(epsg=32645)
tracks_metric = bob_tracks.to_crs(epsg=32645)

print("\n--- Phase 2 Complete ---")
print("New Hospitals CRS:", hospitals_metric.crs.name)
print("New Tracks CRS   :", tracks_metric.crs.name)

Starting Phase 2: Spatial Engine Setup...

✅ Filtered Hospitals: 7697 facilities isolated in WB and Odisha.
✅ Filtered Tracks: 41154 historical storm segments isolated

Reprojecting coordinates to metric... (this takes a few seconds)

--- Phase 2 Complete ---
New Hospitals CRS: WGS 84 / UTM zone 45N
New Tracks CRS   : WGS 84 / UTM zone 45N


In [7]:
tracks_gdf.columns

Index(['SID', 'NUMBER', 'BASIN', 'SUBBASIN', 'ISO_TIME', 'NATURE', 'LAT',
       'LON', 'WMO_WIND', 'WMO_PRES', 'geometry'],
      dtype='str')

In [8]:
print("--- Phase 3a: Hazard Modelling (Amphan) ---")


start_date = '2020-05-16'
end_date = '2020-05-21'

amphan_tracks = tracks_metric[
    (tracks_metric['ISO_TIME'] >= start_date) &
    (tracks_metric['ISO_TIME']<= end_date)
    ].copy()

total_segments = len(amphan_tracks)
missing_winds = amphan_tracks['WMO_WIND'].isna().sum()

print(f"Total Amphan track segments: {total_segments}")
print(f"Segments missing exact wind speed: {missing_winds}\n")

amphan_tracks['WMO_WIND'] = amphan_tracks['WMO_WIND'].fillna(34)

print("Drawing 50km wind footprints around storm track...")
amphan_wind_zones = amphan_tracks.copy()
amphan_wind_zones['geometry'] = amphan_wind_zones.geometry.buffer(50000)

print("Intersecting hospitals with wind zones...")
affected_hospitals = gpd.sjoin(
    hospitals_metric,
    amphan_wind_zones,
    how = "inner",
    predicate = "intersects"
)
print(f"✅ Total facilities inside the 50km danger zone: {len(affected_hospitals)}\n")

--- Phase 3a: Hazard Modelling (Amphan) ---
Total Amphan track segments: 41
Segments missing exact wind speed: 0

Drawing 50km wind footprints around storm track...
Intersecting hospitals with wind zones...
✅ Total facilities inside the 50km danger zone: 2870



In [9]:
print("--- Phase 3b: Vulnerability Modelling (Amphan) ---")

import numpy as np

def calculate_mdr(wind_speed_knots):
    v_half = 100
    k = 0.05
    mdr = 1 / (1 + np.exp(-k * (wind_speed_knots - v_half)))
    return round(mdr, 4)

affected_hospitals['MDR'] = affected_hospitals['WMO_WIND'].apply(calculate_mdr)

columns_to_view = ['adm2_name', 'name_latin', 'WMO_WIND', 'MDR']
top_damage = affected_hospitals[columns_to_view].sort_values(by = 'MDR', ascending = False)

print("--- Highest Risk Facilities ---")
top_damage.head(10)

--- Phase 3b: Vulnerability Modelling (Amphan) ---
--- Highest Risk Facilities ---


,adm2_name,name_latin,WMO_WIND,MDR
5433,South Twenty Four Parganas,Drbachati Sub Health Centre,90.0,0.3775
133032,South Twenty Four Parganas,Matherdighi BPHC,90.0,0.3775
133033,South Twenty Four Parganas,Simultala Hospital,90.0,0.3775
133034,South Twenty Four Parganas,"PHC, Kanthalberia",90.0,0.3775
133035,South Twenty Four Parganas,Canning Sub-Divisional Hospital,90.0,0.3775
133036,South Twenty Four Parganas,Deepanjan Nursing Home,90.0,0.3775
5456,South Twenty Four Parganas,Mathurapur Rural Hospital,90.0,0.3775
5441,South Twenty Four Parganas,Kulpi Rural Hospital,90.0,0.3775
5442,South Twenty Four Parganas,"Maa Durga Nursing Home, South 24 Parganas",90.0,0.3775
5443,South Twenty Four Parganas,"PHC, Ramkishorepur",90.0,0.3775


In [10]:
import pandas as pd

print("--- Phase 3c: Multi-Storm Batch Processing ---")

storm_catalog = [
    {'event_id': 'Odisha_1999', 'start': '1999-10-25', 'end': '1999-11-04'},
    {'event_id': 'Aila_2009',   'start': '2009-05-23', 'end': '2009-05-26'},
    {'event_id': 'Fani_2019',   'start': '2019-04-26', 'end': '2019-05-04'},
    {'event_id': 'Amphan_2020', 'start': '2020-05-16', 'end': '2020-05-21'},
    {'event_id': 'Yaas_2021',   'start': '2021-05-23', 'end': '2021-05-28'},
    {'event_id': 'Dana_2024',   'start': '2024-10-22', 'end': '2024-10-26'}
]

master_loss_list = []

for storm in storm_catalog:
    print(f"Processing {storm['event_id']}...")

    tracks = tracks_metric[(tracks_metric['ISO_TIME'] >= storm['start']) & 
                           (tracks_metric['ISO_TIME'] <= storm['end'])].copy()  

    tracks['WMO_WIND'] = tracks['WMO_WIND'].fillna(34)

    wind_zones = tracks.copy()
    wind_zones['geometry'] = wind_zones.geometry.buffer(50000)

    affected = gpd.sjoin(hospitals_metric, wind_zones, how="inner", predicate="intersects")

    affected['Event_Name'] = storm['event_id']

    affected['MDR'] = affected['WMO_WIND'].apply(calculate_mdr)

    master_loss_list.append(affected)

master_affected_hospitals = pd.concat(master_loss_list, ignore_index=True) 

columns_for_db = ['Event_Name','id', 'name_latin', 'adm1_name', 'adm2_name', 'WMO_WIND', 'MDR']
final_sql_export = master_affected_hospitals.drop(columns=['geometry'])[columns_for_db]

csv_filename = "historical_cyclone_losses.csv"
final_sql_export.to_csv(csv_filename, index=False)

print(f"✅ Phase 3 officially complete. {len(final_sql_export)} total damage records exported to {csv_filename} for SQL.")

--- Phase 3c: Multi-Storm Batch Processing ---
Processing Odisha_1999...
Processing Aila_2009...
Processing Fani_2019...
Processing Amphan_2020...
Processing Yaas_2021...
Processing Dana_2024...
✅ Phase 3 officially complete. 31500 total damage records exported to historical_cyclone_losses.csv for SQL.
